# UniDatabase — Python quickstart

`unidatabase` is a Cython extension over the UniDatabase C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unidatabase
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import tempfile
from pathlib import Path

import unidatabase

unidatabase.version(), unidatabase.abi_version()

('0.1.0', 1)

## A database

`Database` opens a SQLite file and is a context manager: leaving the block
closes the connection. The parent directory is created if it is missing.

In [2]:
directory = tempfile.TemporaryDirectory()
path = str(Path(directory.name) / "quickstart.sqlite")

db = unidatabase.Database(path)
db.execute("CREATE TABLE note(id INTEGER PRIMARY KEY, text TEXT)")
db.execute("INSERT INTO note(text) VALUES ('kept')")
db.is_open

True

## Closing

`close()` is idempotent — calling it twice is not an error, which matters
because the C ABI owns the handle and a double free would not be recoverable.

In [3]:
db.close()
db.close()
db.is_open

False

Using a closed database raises rather than failing quietly:

In [4]:
try:
    db.execute("SELECT 1")
except RuntimeError as exc:
    print("RuntimeError:", exc)

RuntimeError: the database is closed


## Errors carry SQLite's own message

The C ABI reports a failure as a false return and leaves the reason in its own
error slot; the binding reads it before the next call can overwrite it, and
raises with it.

In [5]:
with unidatabase.Database(path) as db:
    try:
        db.execute("SELECT * FROM a_table_that_does_not_exist")
    except RuntimeError as exc:
        print("RuntimeError:", exc)

RuntimeError: no such table: a_table_that_does_not_exist


A path that is not a string is a type error, not a coercion.

In [6]:
try:
    unidatabase.Database(42)
except TypeError as exc:
    print("TypeError:", exc)

TypeError: path must be str, got int


In [7]:
directory.cleanup()

## The C ABI underneath

The same engine is reachable from anything that speaks C, handle-based:

```c
void *unidatabase_open(const char *path);
int   unidatabase_execute(void *connection, const char *sql);
void  unidatabase_close(void *connection);
const char *unidatabase_last_error(void);
```

There a failure is a NULL or zero return with the reason in
`unidatabase_last_error`, because an exception must never unwind across an ABI
boundary.

See `include/UniDatabase.h`, and the book for the full picture.